In [18]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
NUM_FRAMES = 15
BATCH_SIZE = 4
EPOCHS = 10

In [19]:
def load_video_frames(video_path):
    frames = sorted(os.listdir(video_path))[:NUM_FRAMES]
    video = []

    for frame in frames:
        img_path = os.path.join(video_path, frame)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        video.append(img)

    # Padding if less frames
    while len(video) < NUM_FRAMES:
        video.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(video, dtype=np.float32)


def generator(root_dir):
    classes = ["Celeb-real", "Celeb-synthesis"]

    for label, cls in enumerate(classes):
        class_path = os.path.join(root_dir, cls)

        for video_folder in os.listdir(class_path):
            video_path = os.path.join(class_path, video_folder)

            if os.path.isdir(video_path):
                video_tensor = load_video_frames(video_path)
                yield video_tensor, label

In [20]:
output_signature = (
    tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator("dataset_faces/dataset_faces/train"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: generator("dataset_faces/dataset_faces/val"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_generator(
    lambda: generator("dataset_faces/dataset_faces/test"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [21]:
def build_model():

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg"
    )
    backbone.trainable = False

    video_input = layers.Input(
        shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3)
    )

    # Spatial features
    x = layers.TimeDistributed(backbone)(video_input)
    # shape: (batch, NUM_FRAMES, 1280)

    # ---- Temporal Transformer ----

    x1 = layers.LayerNormalization()(x)

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=128
    )(x1, x1)

    x2 = layers.Add()([x, attention])   # OK same shape

    x3 = layers.LayerNormalization()(x2)

    ff = layers.Dense(512, activation="relu")(x3)
    ff = layers.Dense(1280)(ff)   # 🔥 IMPORTANT: back to 1280

    x4 = layers.Add()([x2, ff])   # Now shapes match

    x = layers.GlobalAveragePooling1D()(x4)

    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(video_input, output)

    return model

In [ ]:
model = build_model()
model.summary()

In [25]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

Epoch 1/10


UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset_faces/dataset_faces/train\\Celeb-real'
Traceback (most recent call last):

  File "c:\Users\praveen tamminaina\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "c:\Users\praveen tamminaina\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "c:\Users\praveen tamminaina\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "C:\Users\praveen tamminaina\AppData\Local\Temp\ipykernel_7636\122066308.py", line 25, in generator
    

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
def make_gradcam_heatmap(model, video_tensor, layer_name="top_conv"):

    # Take one frame only for GradCAM
    frame = video_tensor[0:1, 0]  # first frame

    backbone = model.layers[1].layer  # EfficientNet

    grad_model = keras.Model(
        [backbone.inputs],
        [backbone.get_layer(layer_name).output, backbone.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(frame)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
import matplotlib.pyplot as plt

sample_video, label = next(iter(test_dataset))
heatmap = make_gradcam_heatmap(model, sample_video)

plt.matshow(heatmap)
plt.colorbar()
plt.show()